# Enhanced Grad-CAM for six-emotion FER — full run on Colab
Runtime → Change runtime type → **T4 GPU**. Run the cells in order. Every result is written straight to Google Drive (`MyDrive/fer_msf_gradcam_runs`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/fer_msf_gradcam_runs'
!mkdir -p $OUT/runs

In [ ]:
%cd /content
!rm -rf fer-enhanced-gradcam
!git clone -q https://github.com/Nadi-989/fer-enhanced-gradcam.git
%cd fer-enhanced-gradcam
!pip install -q -r requirements.txt
!rm -rf runs && ln -s $OUT/runs runs
!nvidia-smi --query-gpu=name --format=csv,noheader

## Data
The cell looks for the datasets in your Drive (`MyDrive/fer_data`, or anywhere in MyDrive), then tries to **download them from Kaggle automatically**, and only if that fails shows an **upload button**.
- **FER2013**: `fer2013.csv`, or the Kaggle zip (msambare/fer2013)
- **CK+**: `ckextended.csv`, or the zip that contains it

In [ ]:
!pip install -q kagglehub
import os, glob, zipfile, shutil, subprocess
DATA = '/content/drive/MyDrive/fer_data'
os.makedirs(DATA, exist_ok=True)

def candidates():
    pats = [f'{DATA}/*', '/content/drive/MyDrive/*', '/content/drive/MyDrive/*/*']
    return sorted({p for pat in pats for p in glob.glob(pat) if os.path.isfile(p)})

def find_data():
    fer_csv = fer_zip = ck_csv = None
    for p in candidates():
        name = os.path.basename(p).lower()
        if name == 'fer2013.csv': fer_csv = fer_csv or p
        elif name == 'ckextended.csv': ck_csv = ck_csv or p
        elif name.endswith('.zip'):
            try: names = zipfile.ZipFile(p).namelist()
            except zipfile.BadZipFile: continue
            if any(n.endswith('ckextended.csv') for n in names) and not ck_csv:
                zipfile.ZipFile(p).extractall('/content/tmp_ck'); ck_csv = glob.glob('/content/tmp_ck/**/ckextended.csv', recursive=True)[0]
            if any(n.endswith('fer2013.csv') for n in names) and not fer_csv:
                zipfile.ZipFile(p).extractall('/content/tmp_fer'); fer_csv = glob.glob('/content/tmp_fer/**/fer2013.csv', recursive=True)[0]
            elif any(n.startswith(('train/', 'test/')) for n in names) and not fer_zip:
                fer_zip = p
    return fer_csv, fer_zip, ck_csv

fer_csv, fer_zip, ck_csv = find_data()

def kaggle_fetch(slug):
    """Try to download a public Kaggle dataset into MyDrive/fer_data (no key needed for most public sets)."""
    try:
        import kagglehub
        path = kagglehub.dataset_download(slug)
        dst = f'{DATA}/{slug.split("/")[1]}.zip'
        shutil.make_archive(dst[:-4], 'zip', path)
        print('Downloaded', slug)
    except Exception as e:
        print(f'Could not download {slug} automatically ({type(e).__name__}).')

if not (fer_csv or fer_zip):
    kaggle_fetch('msambare/fer2013')
if not ck_csv:
    kaggle_fetch('davilsena/ckdataset')
fer_csv, fer_zip, ck_csv = find_data()

if not (fer_csv or fer_zip) or not ck_csv:
    print('Missing:', ('FER2013 ' if not (fer_csv or fer_zip) else '') + ('CK+ (ckextended.csv)' if not ck_csv else ''))
    print('Upload the missing file(s) now (they are also saved to MyDrive/fer_data):')
    from google.colab import files
    for name, data in files.upload().items():
        open(f'{DATA}/{name}', 'wb').write(data)
    fer_csv, fer_zip, ck_csv = find_data()

HAVE_FER = bool(fer_csv or fer_zip); HAVE_CK = bool(ck_csv)
if fer_csv:
    subprocess.run(['python', 'scripts/prepare_csv.py', '--csv', fer_csv, '--out', 'data/fer2013'], check=True)
elif fer_zip:
    shutil.rmtree('data/fer2013', ignore_errors=True); zipfile.ZipFile(fer_zip).extractall('data/fer2013')
if ck_csv:
    subprocess.run(['python', 'scripts/prepare_csv.py', '--csv', ck_csv, '--out', 'data/ckplus48', '--pool'], check=True)
print('FER2013 ready:', HAVE_FER, '| CK+ ready:', HAVE_CK)
if not (HAVE_FER or HAVE_CK):
    raise SystemExit('No dataset found - check MyDrive/fer_data and run this cell again.')

## FER2013
Start with `ARCHS = ['resnet18']` (about 1 hour on T4); add the other two when it works.

In [ ]:
ARCHS = ['resnet18']   # full paper: ['resnet18', 'vgg16', 'efficientnet_b0']
if not HAVE_FER:
    print('FER2013 not found - skipped')
else:
    for arch in ARCHS:
        !python train.py --config configs/fer2013.yaml --arch {arch}
        !python evaluate_cam.py --run runs/fer2013_{arch}
    !python sanity_check.py --run runs/fer2013_resnet18
    !python make_figures.py --run runs/fer2013_resnet18 --qualitative --regions --bars

## CK+ — 5-fold cross-validation

In [ ]:
ARCHS_CK = ['resnet18', 'vgg16', 'efficientnet_b0']
if not HAVE_CK:
    print('CK+ not found - skipped')
else:
    for arch in ARCHS_CK:
        for f in range(5):
            !python train.py --config configs/ckplus48.yaml --arch {arch} --fold {f} --out_dir runs/ckplus48_{arch}/fold{f}
            !python evaluate_cam.py --run runs/ckplus48_{arch}/fold{f}
        !python aggregate_cv.py --runs runs/ckplus48_{arch}/fold*

## Tables for the paper
Everything is already in Drive: `MyDrive/fer_msf_gradcam_runs/runs`.

In [ ]:
!for f in runs/*/cam_summary.md; do echo "== $f"; cat "$f"; done
!for f in runs/*/test_results.json runs/*/cv_results.json; do echo "== $f"; head -8 "$f"; done